# Paso 2 - Hibrido adaptativo jerarquico

Objetivo: convertir el hibrido fijo `Sequential + Category` del H2 en una contribucion mas fuerte.

Cambios principales:

- Usar `category_tree.csv` para hacer backoff jerarquico de categorias.
- Reemplazar pesos fijos por pesos adaptativos basados en confianza de transicion.
- Comparar contra ablations: hibrido fijo, hibrido adaptativo no jerarquico e hibrido adaptativo jerarquico.

In [1]:
from pathlib import Path
from collections import Counter, defaultdict
import math
import sys

import numpy as np
import pandas as pd

PROJECT_DIR = Path("..").resolve().parent
DATA_DIR = PROJECT_DIR / "data"
OUTPUT_DIR = PROJECT_DIR / "H3" / "outputs"
CACHE_DIR = PROJECT_DIR / "H3" / "cache"
SRC_DIR = PROJECT_DIR / "H3" / "src"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(SRC_DIR))

from cache_utils import load_or_build
from step1_metrics_examples import (
    build_category_recommender,
    build_examples,
    build_fixed_hybrid_recommender,
    build_global_popularity,
    build_random_recommender,
    build_seen_items,
    build_train_sequences,
    evaluate_models,
    filter_seen,
    load_events,
    load_latest_item_categories,
    rank_to_scores,
    sample_eval_users,
    temporal_leave_one_out,
)

K = 10
SEED = 42
MAX_EVAL_USERS = 50_000  # use 0 for the final full run
FORCE_REBUILD = False

## 1. Cargar split y metadata

In [2]:
events = load_or_build(
    CACHE_DIR / "events_preprocessed.pkl",
    lambda: load_events(DATA_DIR),
    force=FORCE_REBUILD,
)
train, test = load_or_build(
    CACHE_DIR / "temporal_leave_one_out_split.pkl",
    lambda: temporal_leave_one_out(events),
    force=FORCE_REBUILD,
)
eval_users = sample_eval_users(test, MAX_EVAL_USERS, SEED)

item_categories = load_or_build(
    CACHE_DIR / "latest_item_categories.pkl",
    lambda: load_latest_item_categories(DATA_DIR),
    force=FORCE_REBUILD,
)
item_to_category = load_or_build(
    CACHE_DIR / "item_to_category.pkl",
    lambda: dict(zip(item_categories["itemid"].astype(int), item_categories["categoryid"].astype(int))),
    force=FORCE_REBUILD,
)

seen_items = load_or_build(
    CACHE_DIR / "seen_items.pkl",
    lambda: build_seen_items(train),
    force=FORCE_REBUILD,
)
train_sequences = load_or_build(
    CACHE_DIR / "train_sequences.pkl",
    lambda: build_train_sequences(train),
    force=FORCE_REBUILD,
)
train_history_lengths = load_or_build(
    CACHE_DIR / "train_history_lengths.pkl",
    lambda: train.groupby("visitorid").size().astype(int).to_dict(),
    force=FORCE_REBUILD,
)
fallback_items, item_prob, item_popularity_score = load_or_build(
    CACHE_DIR / "global_popularity.pkl",
    lambda: build_global_popularity(train),
    force=FORCE_REBUILD,
)
catalog_items = load_or_build(
    CACHE_DIR / "catalog_items.pkl",
    lambda: sorted(train["itemid"].astype(int).unique().tolist()),
    force=FORCE_REBUILD,
)
catalog_size = len(catalog_items)

print(f"Train events: {len(train):,}")
print(f"Evaluable users: {test['visitorid'].nunique():,}")
print(f"Users evaluated in this run: {len(eval_users):,}")
print(f"Catalog size: {catalog_size:,}")
print(f"Items with category: {len(item_to_category):,}")

Train events: 1,348,521
Evaluable users: 406,020
Users evaluated in this run: 50,000
Catalog size: 150,790
Items with category: 417,053


## 2. Utilidades para la jerarquia de categorias

La jerarquia se representa como `categoryid -> parentid`. Para cada categoria de item recuperamos su camino de ancestros y lo usamos como backoff semantico cuando la senal de categoria exacta es escasa.

In [3]:
def load_category_parent_map(data_dir: Path) -> dict[int, int | None]:
    category_tree = pd.read_csv(data_dir / "category_tree.csv")
    parent_map = {}
    for row in category_tree.itertuples(index=False):
        category = int(row.categoryid)
        parent = None if pd.isna(row.parentid) else int(row.parentid)
        parent_map[category] = parent
    return parent_map


def ancestor_path(category: int, parent_map: dict[int, int | None]) -> list[int]:
    path = []
    seen = set()
    current = int(category)
    while current is not None and current not in seen:
        path.append(current)
        seen.add(current)
        current = parent_map.get(current)
    return path


parent_map = load_or_build(
    CACHE_DIR / "category_parent_map.pkl",
    lambda: load_category_parent_map(DATA_DIR),
    force=FORCE_REBUILD,
)
category_to_ancestors = load_or_build(
    CACHE_DIR / "category_to_ancestors.pkl",
    lambda: {category: ancestor_path(category, parent_map) for category in parent_map},
    force=FORCE_REBUILD,
)

depths = [len(path) - 1 for path in category_to_ancestors.values()]
print(f"Categories: {len(parent_map):,}")
print(f"Roots: {sum(parent is None for parent in parent_map.values()):,}")
print(f"Median depth: {np.median(depths):.1f}; max depth: {max(depths)}")

Categories: 1,669
Roots: 25
Median depth: 2.0; max depth: 5


## 3. Recomendador secuencial con confianza de transicion

El gate adaptativo necesita una senal de confianza. Usamos la cantidad de evidencia de transiciones aprendida para el ultimo item del usuario en train. Si el ultimo item tiene muchas transiciones salientes, se confia mas en el componente secuencial; si no, suben de peso los componentes de categoria y jerarquia.

In [4]:
def build_transition_artifacts(train):
    transition_counts = defaultdict(Counter)
    for _, user_df in train.groupby("visitorid", sort=False):
        items = [int(item) for item in user_df["itemid"].tolist()]
        weights = [float(weight) for weight in user_df["event_weight"].tolist()]
        for previous_item, next_item, next_weight in zip(items[:-1], items[1:], weights[1:]):
            if previous_item != next_item:
                transition_counts[previous_item][next_item] += next_weight

    transition_rankings = {
        item: [next_item for next_item, _ in counts.most_common(300)]
        for item, counts in transition_counts.items()
    }
    transition_strength = {
        item: float(sum(counts.values()))
        for item, counts in transition_counts.items()
    }
    strengths = np.array(list(transition_strength.values()), dtype=float)
    confidence_scale = float(np.quantile(strengths, 0.95)) if len(strengths) else 1.0
    confidence_scale = max(confidence_scale, 1.0)
    return transition_rankings, transition_strength, confidence_scale


def build_transition_recommender_with_confidence(train, train_sequences, seen_items, fallback_items):
    transition_rankings, transition_strength, confidence_scale = load_or_build(
        CACHE_DIR / "transition_artifacts.pkl",
        lambda: build_transition_artifacts(train),
        force=FORCE_REBUILD,
    )

    def transition_confidence(user: int) -> float:
        sequence = train_sequences.get(user, [])
        if not sequence:
            return 0.0
        last_item = int(sequence[-1])
        strength = transition_strength.get(last_item, 0.0)
        return float(min(1.0, math.log1p(strength) / math.log1p(confidence_scale)))

    def recommend(user: int, k: int) -> list[int]:
        seen = seen_items.get(user, set())
        sequence = train_sequences.get(user, [])
        candidates = []
        if sequence:
            candidates.extend(transition_rankings.get(int(sequence[-1]), []))
        candidates.extend(fallback_items)
        return filter_seen(candidates, seen, k)

    return recommend, transition_confidence

## 4. Recomendador jerarquico por categoria

Este recomendador usa primero la ultima categoria exacta del usuario. Si esa categoria es demasiado estrecha o no es suficientemente informativa, hace backoff por sus ancestros antes de caer en popularidad global.

In [5]:
def build_hierarchical_category_artifacts(
    train,
    item_to_category,
    category_to_ancestors,
    train_sequences,
    max_items_per_level=300,
):
    train_with_category = train.copy()
    train_with_category["categoryid"] = train_with_category["itemid"].map(item_to_category)
    train_with_category = train_with_category.dropna(subset=["categoryid"]).copy()
    train_with_category["categoryid"] = train_with_category["categoryid"].astype(int)

    level_scores = defaultdict(Counter)
    for row in train_with_category[["itemid", "categoryid", "event_weight"]].itertuples(index=False):
        item = int(row.itemid)
        category = int(row.categoryid)
        weight = float(row.event_weight)
        path = category_to_ancestors.get(category, [category])
        for depth, ancestor in enumerate(path):
            # Exact category keeps full weight; ancestors receive decayed evidence.
            level_scores[int(ancestor)][item] += weight / (depth + 1)

    category_level_rankings = {
        category: [item for item, _ in scores.most_common(max_items_per_level)]
        for category, scores in level_scores.items()
    }

    last_user_category = {}
    for user, sequence in train_sequences.items():
        for item in reversed(sequence):
            category = item_to_category.get(int(item))
            if category is not None:
                last_user_category[int(user)] = int(category)
                break

    return category_level_rankings, last_user_category


def build_hierarchical_category_recommender(
    train,
    item_to_category,
    category_to_ancestors,
    train_sequences,
    seen_items,
    fallback_items,
    min_candidates_per_level=20,
    max_items_per_level=300,
):
    category_level_rankings, last_user_category = load_or_build(
        CACHE_DIR / "hierarchical_category_artifacts.pkl",
        lambda: build_hierarchical_category_artifacts(
            train,
            item_to_category,
            category_to_ancestors,
            train_sequences,
            max_items_per_level=max_items_per_level,
        ),
        force=FORCE_REBUILD,
    )

    def candidate_path(user: int) -> list[int]:
        category = last_user_category.get(int(user))
        if category is None:
            return []
        return category_to_ancestors.get(category, [category])

    def recommend(user: int, k: int) -> list[int]:
        seen = seen_items.get(user, set())
        candidates = []
        for category in candidate_path(user):
            level_items = category_level_rankings.get(int(category), [])
            candidates.extend(level_items)
            if len(filter_seen(candidates, seen, k=min_candidates_per_level)) >= min_candidates_per_level:
                break
        candidates.extend(fallback_items)
        return filter_seen(candidates, seen, k)

    return recommend, candidate_path

## 5. Hibridos adaptativos

Dos ablations son utiles para el paper:

- Hibrido adaptativo no jerarquico: pesos adaptativos, solo categoria directa.
- Hibrido adaptativo jerarquico: pesos adaptativos mas backoff jerarquico de categorias.

In [6]:
def adaptive_weights(transition_conf: float, history_length: int) -> dict[str, float]:
    # History length gives mild support to the sequential signal, but transition evidence dominates.
    history_conf = min(1.0, math.log1p(max(history_length, 0)) / math.log1p(10))
    seq_conf = 0.75 * transition_conf + 0.25 * history_conf

    seq = 0.20 + 0.60 * seq_conf
    category = 0.55 - 0.30 * seq_conf
    hierarchy = 0.20 * (1.0 - transition_conf)
    popularity = 0.05
    total = seq + category + hierarchy + popularity
    return {
        "seq": seq / total,
        "category": category / total,
        "hierarchy": hierarchy / total,
        "popularity": popularity / total,
    }


def build_adaptive_hybrid_recommender(
    sequential_recommender,
    transition_confidence,
    category_recommender,
    hierarchical_recommender,
    seen_items,
    fallback_items,
    train_history_lengths,
    use_hierarchy: bool,
):
    def recommend(user: int, k: int) -> list[int]:
        t_conf = transition_confidence(user)
        h_len = int(train_history_lengths.get(user, 0))
        weights = adaptive_weights(t_conf, h_len)

        scores = defaultdict(float)
        for item, score in rank_to_scores(sequential_recommender(user, 200), weights["seq"], 200).items():
            scores[item] += score
        for item, score in rank_to_scores(category_recommender(user, 200), weights["category"], 200).items():
            scores[item] += score
        if use_hierarchy:
            for item, score in rank_to_scores(hierarchical_recommender(user, 200), weights["hierarchy"], 200).items():
                scores[item] += score
        for item, score in rank_to_scores(fallback_items[:200], weights["popularity"], 200).items():
            scores[item] += score

        ranked = [item for item, _ in sorted(scores.items(), key=lambda pair: (-pair[1], pair[0]))]
        return filter_seen(ranked + fallback_items, seen_items.get(user, set()), k)

    return recommend

## 6. Construir modelos

In [7]:
random_recommender = build_random_recommender(catalog_items, seen_items, SEED)
most_popular_recommender = lambda user, k: filter_seen(fallback_items, seen_items.get(user, set()), k)

sequential_recommender, transition_confidence = build_transition_recommender_with_confidence(
    train, train_sequences, seen_items, fallback_items
)
category_recommender = build_category_recommender(
    train, item_to_category, train_sequences, seen_items, fallback_items
)
hierarchical_recommender, hierarchical_path = build_hierarchical_category_recommender(
    train,
    item_to_category,
    category_to_ancestors,
    train_sequences,
    seen_items,
    fallback_items,
)
fixed_hybrid_recommender = build_fixed_hybrid_recommender(
    sequential_recommender, category_recommender, seen_items, fallback_items
)
adaptive_hybrid_recommender = build_adaptive_hybrid_recommender(
    sequential_recommender,
    transition_confidence,
    category_recommender,
    hierarchical_recommender,
    seen_items,
    fallback_items,
    train_history_lengths,
    use_hierarchy=False,
)
adaptive_hierarchical_hybrid_recommender = build_adaptive_hybrid_recommender(
    sequential_recommender,
    transition_confidence,
    category_recommender,
    hierarchical_recommender,
    seen_items,
    fallback_items,
    train_history_lengths,
    use_hierarchy=True,
)

models = {
    "Random": random_recommender,
    "Most Popular": most_popular_recommender,
    "Sequential Transition": sequential_recommender,
    "Category Popularity": category_recommender,
    "Hierarchical Category": hierarchical_recommender,
    "Fixed Hybrid Seq+Category": fixed_hybrid_recommender,
    "Adaptive Hybrid": adaptive_hybrid_recommender,
    "Adaptive Hierarchical Hybrid": adaptive_hierarchical_hybrid_recommender,
}

list(models)

['Random',
 'Most Popular',
 'Sequential Transition',
 'Category Popularity',
 'Hierarchical Category',
 'Fixed Hybrid Seq+Category',
 'Adaptive Hybrid',
 'Adaptive Hierarchical Hybrid']

## 7. Evaluar y guardar outputs

In [8]:
detailed, summary, coverage_items = evaluate_models(
    models,
    eval_users,
    test,
    train_history_lengths,
    item_prob,
    item_to_category,
    catalog_size,
    K,
)

short_summary = (
    detailed[detailed["history_length"] <= 2]
    .groupby("model")[[f"precision@{K}", f"recall@{K}", f"ndcg@{K}", f"novelty@{K}", f"category_diversity@{K}"]]
    .mean()
    .reset_index()
    .sort_values(f"recall@{K}", ascending=False)
)

examples = build_examples(eval_users, train, test, models, item_to_category, K)

suffix = "full" if MAX_EVAL_USERS == 0 else f"sample_{len(eval_users)}"
summary_path = OUTPUT_DIR / f"step2_metrics_summary_{suffix}.csv"
short_path = OUTPUT_DIR / f"step2_metrics_short_history_{suffix}.csv"
detailed_path = OUTPUT_DIR / f"step2_metrics_detailed_{suffix}.csv"
examples_path = OUTPUT_DIR / f"step2_qualitative_examples_{suffix}.csv"

summary.to_csv(summary_path, index=False)
short_summary.to_csv(short_path, index=False)
detailed.to_csv(detailed_path, index=False)
examples.to_csv(examples_path, index=False)

display(summary)
display(short_summary)
display(examples.head(24))

summary_path, short_path, detailed_path, examples_path

,model,precision@10,recall@10,ndcg@10,novelty@10,category_diversity@10,catalog_coverage@10
0,Adaptive Hierarchical Hybrid,0.011274,0.11274,0.068430,13.549909,0.514987,0.298004
1,Adaptive Hybrid,0.011202,0.11202,0.068732,13.470180,0.544921,0.299436
3,Fixed Hybrid Seq+Category,0.010862,0.10862,0.068026,13.166489,0.622498,0.298820
7,Sequential Transition,0.010154,0.10154,0.065877,13.245488,0.729615,0.328835
2,Category Popularity,0.007654,0.07654,0.041117,13.504187,0.070672,0.065833
4,Hierarchical Category,0.007608,0.07608,0.040924,13.511239,0.072520,0.065581
5,Most Popular,0.000226,0.00226,0.000947,10.197979,0.955462,0.000133
6,Random,0.000004,0.00004,0.000026,18.788678,0.995118,0.963565


,model,precision@10,recall@10,ndcg@10,novelty@10,category_diversity@10
0,Adaptive Hierarchical Hybrid,0.012525,0.125250,0.076557,13.468441,0.514507
1,Adaptive Hybrid,0.012451,0.124507,0.076941,13.391529,0.544144
3,Fixed Hybrid Seq+Category,0.012031,0.120310,0.076247,13.052645,0.639625
7,Sequential Transition,0.011203,0.112028,0.073790,13.095343,0.742681
2,Category Popularity,0.008564,0.085642,0.046195,13.455683,0.090289
4,Hierarchical Category,0.008516,0.085156,0.045961,13.461435,0.092380
5,Most Popular,0.000251,0.002513,0.001079,10.197691,0.955510
6,Random,0.000003,0.000029,0.000009,18.789946,0.995071


,visitorid,history_length,recent_history,recent_categories,model,recommendations,recommendation_categories,true_item,true_category,hit@k
0,513229,1,view:264876,[1047],Random,"[50443, 154365, 82034, 11101, 302335, 264945, ...","[196, 1558, 1263, 1343, 65, None, 491, 491, 13...",290993,484,0
1,513229,1,view:264876,[1047],Most Popular,"[461686, 257040, 119736, 320130, 9877, 309778,...","[1037, 683, 57, 1483, 858, 683, 1098, 5, 398, 5]",290993,484,0
2,513229,1,view:264876,[1047],Sequential Transition,"[126236, 340490, 290993, 406795, 463165, 20249...","[1047, 50, 484, 1047, 1047, 884, 1355, 1355, 1...",290993,484,1
3,513229,1,view:264876,[1047],Category Popularity,"[126236, 6692, 297513, 109924, 2992, 404591, 1...","[1047, 1047, 1047, 1047, 1047, 1047, 1047, 104...",290993,484,0
4,513229,1,view:264876,[1047],Hierarchical Category,"[126236, 6692, 297513, 109924, 404591, 2992, 1...","[1047, 1047, 1047, 1047, 1047, 1047, 1047, 104...",290993,484,0
5,513229,1,view:264876,[1047],Fixed Hybrid Seq+Category,"[126236, 340490, 290993, 406795, 463165, 6692,...","[1047, 50, 484, 1047, 1047, 1047, 884, 1037, 1...",290993,484,1
6,513229,1,view:264876,[1047],Adaptive Hybrid,"[126236, 340490, 290993, 6692, 406795, 463165,...","[1047, 50, 484, 1047, 1047, 1047, 1047, 884, 1...",290993,484,1
7,513229,1,view:264876,[1047],Adaptive Hierarchical Hybrid,"[126236, 340490, 290993, 6692, 406795, 463165,...","[1047, 50, 484, 1047, 1047, 1047, 1047, 884, 1...",290993,484,1
8,1223320,1,view:450627,[1388],Random,"[155295, 437602, 346605, 122535, 449752, 30087...","[1476, 1172, 1412, 1526, 1578, 575, 353, None,...",450627,1388,0
9,1223320,1,view:450627,[1388],Most Popular,"[461686, 257040, 119736, 320130, 9877, 309778,...","[1037, 683, 57, 1483, 858, 683, 1098, 5, 398, 5]",450627,1388,0


(PosixPath('/Users/ianfuenzalida/Library/CloudStorage/OneDrive-UniversidadCatólicadeChile/Documentos/Semestre 11/Sistemas Recomendadores/Archivos de Paula Antonia Elicer García - Proyecto SR/H3/outputs/step2_metrics_summary_sample_50000.csv'),
 PosixPath('/Users/ianfuenzalida/Library/CloudStorage/OneDrive-UniversidadCatólicadeChile/Documentos/Semestre 11/Sistemas Recomendadores/Archivos de Paula Antonia Elicer García - Proyecto SR/H3/outputs/step2_metrics_short_history_sample_50000.csv'),
 PosixPath('/Users/ianfuenzalida/Library/CloudStorage/OneDrive-UniversidadCatólicadeChile/Documentos/Semestre 11/Sistemas Recomendadores/Archivos de Paula Antonia Elicer García - Proyecto SR/H3/outputs/step2_metrics_detailed_sample_50000.csv'),
 PosixPath('/Users/ianfuenzalida/Library/CloudStorage/OneDrive-UniversidadCatólicadeChile/Documentos/Semestre 11/Sistemas Recomendadores/Archivos de Paula Antonia Elicer García - Proyecto SR/H3/outputs/step2_qualitative_examples_sample_50000.csv'))

## 8. Inspeccionar pesos adaptativos

Esta tabla sirve para explicar el modelo en el paper: usuarios con historial corto o baja confianza de transicion deberian recibir mas peso de categoria/jerarquia.

In [9]:
weight_rows = []
for user in eval_users[:5000]:
    t_conf = transition_confidence(user)
    h_len = train_history_lengths.get(user, 0)
    row = {"visitorid": user, "history_length": h_len, "transition_confidence": t_conf}
    row.update(adaptive_weights(t_conf, h_len))
    row["hierarchical_path"] = hierarchical_path(user)
    weight_rows.append(row)

weights_df = pd.DataFrame(weight_rows)
display(weights_df.head())
display(weights_df.groupby(pd.cut(weights_df["history_length"], bins=[0, 1, 2, 5, 10, 10_000]))[["seq", "category", "hierarchy", "popularity"]].mean())

,visitorid,history_length,transition_confidence,seq,category,hierarchy,popularity,hierarchical_path
0,513229,1,0.812690,0.584522,0.331541,0.035952,0.047985,"[1047, 127, 312, 653]"
1,1223320,1,0.791985,0.575867,0.336179,0.039946,0.048009,"[1388, 1249, 1482]"
2,647019,1,0.482995,0.445664,0.405944,0.100025,0.048367,"[1246, 945, 653]"
3,1178364,1,0.208015,0.328146,0.468912,0.154251,0.048691,"[333, 1497, 1490]"
4,115515,1,0.329695,0.380342,0.440945,0.130166,0.048547,"[1694, 1621, 395]"


,seq,category,hierarchy,popularity
history_length,,,,
"(0, 1]",0.482229,0.386352,0.083153,0.048267
"(1, 2]",0.542844,0.346959,0.062630,0.047568
"(2, 5]",0.580377,0.319761,0.052943,0.046919
"(5, 10]",0.628770,0.284397,0.040773,0.046060
"(10, 10000]",0.633954,0.277124,0.043221,0.045700
